# 2025-10-20: Process final BMMC objects (whole and per-lineage)
### By [Aishwarya Chander](aishwarya.chander@alleninstitute.org), High Resolution Translational Immunology, Allen Institute for Immunology
**Main aim**: Re-run the full scanpy processing pipeline on the corrected raw object three ways: (1) without Harmony, (2) with Harmony batch correction on sampleKitGuid, and (3) per-L1 lineage subset with and without Harmony. Generates UMAP/tSNE embeddings and ranked genes for each version.

In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import scanpy.external as sce

sc.settings.n_jobs = 30
sc.settings.verbosity = 0

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
def process_adata(adata, 
                  subset_hvgs=False, 
                  run_harmony=False, 
                  harmony_key=["sample.sampleKitGuid"], 
                  resolution=1.0, 
                  run_rank_genes=False):
    
    if adata.raw is None:
        print("No adata.raw found. Please ensure raw data exists before processing.")
        return None

    # Extract raw and re-assign
    adata = adata.raw.to_adata()
    adata.raw = adata

    # mitochondrial genes
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    # ribosomal genes
    adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
    # hemoglobin genes
    adata.var["hb"] = adata.var_names.str.contains(("^HB[^(P)]"))

    sc.pp.calculate_qc_metrics(
        adata, qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True
    )

    # Normalize and log1p
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)

    # HVG selection
    sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.25)
    # Optional: Subset to HVGs
    if subset_hvgs:
        adata = adata[:, adata.var['highly_variable']].copy()

    # Scale and PCA
    sc.pp.scale(adata, max_value=10, zero_center=False)
    sc.tl.pca(adata, svd_solver="arpack")
    adata.obsm["X_pca_temp"] = adata.obsm["X_pca"]

    # Optional: Harmonize on a metadata parameter
    if run_harmony:
        sce.pp.harmony_integrate(adata, key=harmony_key)
        adata.obsm["X_pca"] = adata.obsm["X_pca_harmony"]

    # Neighbors and dimensionality reduction
    sc.pp.neighbors(adata, n_neighbors=50, use_rep="X_pca", n_pcs=20)
    sc.tl.tsne(adata, n_pcs=20)
    sc.tl.umap(adata, min_dist=0.45, random_state=0, n_components=2)
    sc.tl.leiden(adata, resolution=resolution, flavor="igraph", n_iterations=2)

    # Optional: Rank genes (leiden)
    if run_rank_genes:
        sc.tl.rank_genes_groups(
            adata, groupby="leiden", method="t-test", corr_method="benjamini-hochberg"
        )

    return adata

### 1. Process object without Harmony

In [ ]:
adata = sc.read_h5ad('../../../data/rna/final-objects/final-bmmc-raw.h5ad')
adata = process_adata(adata, resolution=0.5, run_rank_genes=True)
adata.write('../../../data/rna/final-objects/final-bmmc-processed.h5ad')

/home/workspace/environment/ndmm-scrna/lib/python3.11/site-packages/scanpy/tools/_tsne.py:126: UserWarning: In previous versions of scanpy, calling tsne with n_jobs > 1 would use MulticoreTSNE. Now this uses the scikit-learn version of TSNE by default. If you'd like the old behaviour (which is deprecated), pass 'use_fast_tsne=True'. Note, MulticoreTSNE is not actually faster anymore.
  warnings.warn(
/home/workspace/environment/ndmm-scrna/lib/python3.11/site-packages/scanpy/tools/_rank_genes_groups.py:452: RuntimeWarning: overflow encountered in expm1
  foldchanges = (self.expm1_func(mean_group) + 1e-9) / (
/home/workspace/environment/ndmm-scrna/lib/python3.11/site-packages/scanpy/tools/_rank_genes_groups.py:452: RuntimeWarning: overflow encountered in expm1
  foldchanges = (self.expm1_func(mean_group) + 1e-9) / (
/home/workspace/environment/ndmm-scrna/lib/python3.11/site-packages/scanpy/tools/_rank_genes_groups.py:452: RuntimeWarning: overflow encountered in expm1
  foldchanges = (sel

### 2. Process object with Harmony

In [ ]:
adata = sc.read_h5ad('../../../data/rna/final-objects/final-bmmc-raw.h5ad')
adata = process_adata(adata, resolution=0.5, run_harmony=True, run_rank_genes=True)
adata.write('../../../data/rna/final-objects/final-bmmc-processed-harmony.h5ad')

### 3. Process L1 clusters with and without harmony

In [ ]:
adata = sc.read_h5ad('../../../data/rna/final-objects/final-bmmc-raw.h5ad')
clusters = list(adata.obs['aifi_celltype_l1'].value_counts().index)
clusters.reverse()

for cluster in clusters:
    print(f'Processing {cluster}')
    subset = adata[adata.obs['aifi_celltype_l1'] == cluster]
    subset = process_adata(subset, resolution=1, run_harmony=True,  run_rank_genes=True)
    subset.write(f'../../../data/rna/bmmc-celltypes/bmmc-{cluster}-processed-harmony.h5ad')


adata = sc.read_h5ad('../../../data/rna/final-objects/final-bmmc-raw.h5ad')
clusters = list(adata.obs['aifi_celltype_l1'].value_counts().index)
clusters.reverse()

for cluster in clusters:
    print(f'Processing {cluster}')
    subset = adata[adata.obs['aifi_celltype_l1'] == cluster]
    subset = process_adata(subset, resolution=1, run_harmony=False,  run_rank_genes=True)
    subset.write(f'../../../data/rna/bmmc-celltypes/bmmc-{cluster}-processed.h5ad')